# 1-Minute → 5-Minute Aggregation: Nautilus INTERNAL vs. Custom Aggregator

Two Nautilus `Strategy` classes that each consume the same 1-minute FX feed and write
5-minute bars to CSV — they differ only in *where* the aggregation happens:

| | Strategy | Subscribes to | Aggregation done by | Output |
|-|----------|---------------|---------------------|--------|
| 1 | `NautilusInternalStrategy` | `5-MINUTE-…-INTERNAL@1-MINUTE-EXTERNAL` | Nautilus DataEngine (built-in) | OHLCV CSV |
| 2 | `CustomAggregationStrategy` | `1-MINUTE-…-EXTERNAL` | `FiveMinAggregator` (this notebook) | OHLCV + derived features CSV |

Strategy 1 just receives 5-min bars in `on_bar` (Nautilus already aggregated them) and
writes them out. Strategy 2 receives 1-min bars, pushes them through a custom
`FiveMinAggregator`, and when the aggregator emits a `Bar5MinFeatures` (a custom
dataclass with OHLCV + derived features) the strategy writes it to CSV.

Trading logic in the strategies is intentionally empty — the goal is to compare the
two aggregation paths.

**Input:** `C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2015/01/02/02.01.2015_ASK_OHLCV.csv`  
**Outputs (project root):** `eurusd_5min_nautilus.csv`, `eurusd_5min_custom_features.csv`

In [ ]:
import csv
import sys
from dataclasses import dataclass, asdict, fields
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the project root robustly. Three search strategies, in order:
#   1. Walk UP from Path.cwd() looking for `core/instrument_factory.py`.
#   2. Walk UP from the notebook's own location.
#   3. Scan immediate children of cwd (VS Code may open an outer workspace).
def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(
        f"could not find project root (no `{marker}`) from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_CSV = r"C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2015/01/02/02.01.2015_ASK_OHLCV.csv"
OUTPUT_NAUTILUS = PROJECT_ROOT / "eurusd_5min_nautilus.csv"
OUTPUT_CUSTOM = PROJECT_ROOT / "eurusd_5min_custom_features.csv"
print(f"cwd         : {Path.cwd()}")
print(f"project root: {PROJECT_ROOT}")

BASE, QUOTE, VENUE_NAME, PRICE_TYPE = "EUR", "USD", "FOREX_MS", "ASK"
OUTPUT_TZ = "Asia/Kolkata"  # render output timestamps in source-file local time (IST, +05:30)

In [2]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig, StrategyConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument

In [3]:
def load_1min_csv(path: str) -> pd.DataFrame:
    """Read the FX 1-min OHLCV CSV → UTC-indexed DataFrame.

    Source format: `DD.MM.YYYY HH:MM:SS.fff GMT+HHMM`.
    """
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S.%f GMT%z", utc=True
    )
    df = df.set_index("timestamp").sort_index()
    return df[["open", "high", "low", "close", "volume"]].astype(float)


df_1min = load_1min_csv(INPUT_CSV)
print(f"loaded {len(df_1min):,} 1-min bars")
print(f"range: {df_1min.index.min()} → {df_1min.index.max()}")
df_1min.head()

loaded 1,230 1-min bars
range: 2015-01-01 22:00:00+00:00 → 2015-01-02 18:29:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2015-01-01 22:00:00+00:00,1.21066,1.21067,1.21059,1.21059,4.50
2015-01-01 22:01:00+00:00,1.21060,1.21067,1.21058,1.21066,15.75
2015-01-01 22:02:00+00:00,1.21061,1.21061,1.21058,1.21058,7.53
2015-01-01 22:03:00+00:00,1.21055,1.21065,1.21049,1.21063,13.53
2015-01-01 22:04:00+00:00,1.21064,1.21071,1.21050,1.21055,44.50


## Custom data class — `Bar5MinFeatures`

What `FiveMinAggregator` emits at the end of each 5-minute window. Plain OHLCV columns plus derived features:

- **Geometry:** `range`, `body`, `upper_wick`, `lower_wick`
- **Returns:** `log_return`, `pct_return` (vs. previous 5-min close)
- **Volume-price:** `typical_price`, `vwap` (cumulative session VWAP)
- **Volatility:** `tr` (true range), `atr_14` (Wilder-style EWM)
- **Trend:** `ema_9`, `ema_21`
- **Momentum:** `rsi_14`
- **Bookkeeping:** `n_ticks` (count of 1-min bars folded into this 5-min bar)

In [4]:
@dataclass
class Bar5MinFeatures:
    """Aggregated 5-minute bar with derived features."""

    timestamp: pd.Timestamp
    open: float
    high: float
    low: float
    close: float
    volume: float
    n_ticks: int
    range: float
    body: float
    upper_wick: float
    lower_wick: float
    log_return: float | None
    pct_return: float | None
    typical_price: float
    vwap: float
    tr: float
    atr_14: float
    ema_9: float
    ema_21: float
    rsi_14: float | None

    @classmethod
    def field_names(cls) -> list[str]:
        return [f.name for f in fields(cls)]

    def to_row(self) -> dict:
        return asdict(self)

## Custom 5-min aggregator — `FiveMinAggregator`

Holds 1-min bars in a buffer until the 5-min boundary is crossed, then folds the
buffer into a single OHLCV bar and computes the derived features. State is updated
incrementally across windows (EMAs, ATR, RSI, cumulative VWAP) so each emit is O(1)
in the number of past bars.

Bucketing convention is **left-open `(T-5, T]`** to match Nautilus' default — a 1-min
bar timestamped exactly on a 5-min boundary closes the prior window.

In [5]:
class FiveMinAggregator:
    """Buffer 1-min bars, emit a `Bar5MinFeatures` at every N-min boundary."""

    def __init__(
        self,
        target_minutes: int = 5,
        ema_short_period: int = 9,
        ema_long_period: int = 21,
        rsi_period: int = 14,
        atr_period: int = 14,
    ) -> None:
        self.target_minutes = target_minutes
        self._alpha_ema_s = 2.0 / (ema_short_period + 1)
        self._alpha_ema_l = 2.0 / (ema_long_period + 1)
        self._alpha_rsi = 1.0 / rsi_period
        self._alpha_atr = 1.0 / atr_period

        # rolling per-window state
        self._buffer: list[Bar] = []
        self._window_close: pd.Timestamp | None = None  # upper bound of current window

        # cross-window state (used to derive features)
        self._prev_close: float | None = None
        self._ema_s: float | None = None
        self._ema_l: float | None = None
        self._avg_gain: float = 0.0
        self._avg_loss: float = 0.0
        self._atr: float | None = None
        self._cum_pv: float = 0.0
        self._cum_v: float = 0.0

    @staticmethod
    def _bar_ts(bar: Bar) -> pd.Timestamp:
        return pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")

    def on_bar(self, bar: Bar) -> Bar5MinFeatures | None:
        """Push a 1-min bar; return a `Bar5MinFeatures` if a window just closed."""
        # Left-open `(T-5, T]` convention to match Nautilus' default: a bar
        # whose timestamp is exactly on a boundary closes the prior window.
        ts = self._bar_ts(bar)
        bucket_close = ts.ceil(f"{self.target_minutes}min")

        emitted: Bar5MinFeatures | None = None
        if self._window_close is None:
            self._window_close = bucket_close
        elif bucket_close != self._window_close:
            emitted = self._close_window()
            self._window_close = bucket_close

        self._buffer.append(bar)
        return emitted

    def flush(self) -> Bar5MinFeatures | None:
        """Emit the trailing partial window (call from `on_stop`)."""
        if self._buffer:
            return self._close_window()
        return None

    def _close_window(self) -> Bar5MinFeatures:
        bars = self._buffer
        opens = float(bars[0].open)
        highs = max(float(b.high) for b in bars)
        lows = min(float(b.low) for b in bars)
        closes = float(bars[-1].close)
        volume = sum(float(b.volume) for b in bars)
        n_ticks = len(bars)

        # geometry
        rng = highs - lows
        body = abs(closes - opens)
        upper_wick = highs - max(opens, closes)
        lower_wick = min(opens, closes) - lows

        # returns vs. previous 5-min close
        if self._prev_close is None or self._prev_close == 0:
            log_ret = None
            pct_ret = None
        else:
            log_ret = float(np.log(closes / self._prev_close))
            pct_ret = closes / self._prev_close - 1.0

        # cumulative VWAP
        typical = (highs + lows + closes) / 3.0
        self._cum_pv += typical * volume
        self._cum_v += volume
        vwap = self._cum_pv / self._cum_v if self._cum_v > 0 else typical

        # true range + Wilder ATR(14)
        if self._prev_close is None:
            tr = rng
        else:
            tr = max(rng, abs(highs - self._prev_close), abs(lows - self._prev_close))
        self._atr = tr if self._atr is None else self._atr + self._alpha_atr * (tr - self._atr)

        # EMAs
        self._ema_s = closes if self._ema_s is None else self._ema_s + self._alpha_ema_s * (closes - self._ema_s)
        self._ema_l = closes if self._ema_l is None else self._ema_l + self._alpha_ema_l * (closes - self._ema_l)

        # RSI(14) — Wilder smoothing of gains/losses
        rsi: float | None = None
        if self._prev_close is not None:
            change = closes - self._prev_close
            gain = max(change, 0.0)
            loss = max(-change, 0.0)
            self._avg_gain += self._alpha_rsi * (gain - self._avg_gain)
            self._avg_loss += self._alpha_rsi * (loss - self._avg_loss)
            if self._avg_loss > 0:
                rs = self._avg_gain / self._avg_loss
                rsi = 100.0 - 100.0 / (1.0 + rs)
            elif self._avg_gain > 0:
                rsi = 100.0

        bar_out = Bar5MinFeatures(
            timestamp=self._window_close,
            open=opens,
            high=highs,
            low=lows,
            close=closes,
            volume=volume,
            n_ticks=n_ticks,
            range=rng,
            body=body,
            upper_wick=upper_wick,
            lower_wick=lower_wick,
            log_return=log_ret,
            pct_return=pct_ret,
            typical_price=typical,
            vwap=vwap,
            tr=tr,
            atr_14=self._atr,
            ema_9=self._ema_s,
            ema_21=self._ema_l,
            rsi_14=rsi,
        )

        self._prev_close = closes
        self._buffer = []
        return bar_out

## Strategy 1 — `NautilusInternalStrategy`

Subscribes to a **composite bar type** of the form
`<id>-5-MINUTE-<price>-INTERNAL@1-MINUTE-EXTERNAL`. The Nautilus `DataEngine`
sees the `INTERNAL@…-EXTERNAL` tail, spins up a `TimeBarAggregator` internally,
and feeds it from the 1-min EXTERNAL bars sitting in the engine's data buffer.
By the time `on_bar` fires, the bar passed in is already a 5-min bar. Strategy
logic = empty; we just append each bar to the output CSV.

In [6]:
class NautilusInternalConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType  # the 5-min INTERNAL@1-MIN-EXTERNAL composite
    output_csv: str


class NautilusInternalStrategy(Strategy):
    """Subscribes to 5-min INTERNAL bars; writes each `on_bar` to CSV."""

    def __init__(self, config: NautilusInternalConfig) -> None:
        super().__init__(config)
        self._fh = None
        self._writer: csv.DictWriter | None = None
        self._n_bars = 0

    def on_start(self) -> None:
        self._fh = open(self.config.output_csv, "w", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(
            self._fh, fieldnames=["timestamp", "open", "high", "low", "close", "volume"]
        )
        self._writer.writeheader()
        self.subscribe_bars(self.config.bar_type)
        self.log.info(f"subscribed to {self.config.bar_type}")

    def on_bar(self, bar: Bar) -> None:
        # bar is already a 5-min bar — Nautilus aggregated it for us
        self._writer.writerow({
            "timestamp": pd.Timestamp(bar.ts_event, unit="ns", tz="UTC").isoformat(),
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume),
        })
        self._n_bars += 1

    def on_stop(self) -> None:
        if self._fh is not None:
            self._fh.close()
        self.log.info(f"wrote {self._n_bars} bars → {self.config.output_csv}")

## Strategy 2 — `CustomAggregationStrategy`

Subscribes to plain `1-MINUTE-…-EXTERNAL` bars. Each `on_bar` is a 1-min bar; the
strategy pushes it into the `FiveMinAggregator`. When the aggregator emits a
`Bar5MinFeatures` (i.e. the previous 5-min window has just closed), the strategy
writes that record to CSV. `on_stop` flushes the trailing partial window.

In [7]:
class CustomAggregationConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType  # the 1-min EXTERNAL bar type
    output_csv: str
    target_minutes: int = 5


class CustomAggregationStrategy(Strategy):
    """Subscribes to 1-min bars; runs them through `FiveMinAggregator`."""

    def __init__(self, config: CustomAggregationConfig) -> None:
        super().__init__(config)
        self._aggregator = FiveMinAggregator(target_minutes=config.target_minutes)
        self._fh = None
        self._writer: csv.DictWriter | None = None
        self._n_bars = 0

    def on_start(self) -> None:
        self._fh = open(self.config.output_csv, "w", newline="", encoding="utf-8")
        self._writer = csv.DictWriter(self._fh, fieldnames=Bar5MinFeatures.field_names())
        self._writer.writeheader()
        self.subscribe_bars(self.config.bar_type)
        self.log.info(f"subscribed to {self.config.bar_type}")

    def _write(self, b5: Bar5MinFeatures) -> None:
        row = b5.to_row()
        row["timestamp"] = b5.timestamp.isoformat()
        self._writer.writerow(row)
        self._n_bars += 1

    def on_bar(self, bar: Bar) -> None:
        b5 = self._aggregator.on_bar(bar)
        if b5 is not None:
            self._write(b5)

    def on_stop(self) -> None:
        trailing = self._aggregator.flush()
        if trailing is not None:
            self._write(trailing)
        if self._fh is not None:
            self._fh.close()
        self.log.info(f"wrote {self._n_bars} bars → {self.config.output_csv}")

## Backtest harness

Both strategies need the same input — 1-minute EXTERNAL bars in the engine's data
buffer. Strategy 1 then asks the engine for 5-min INTERNAL bars (which the engine
synthesises from the 1-min ones); Strategy 2 just consumes the 1-min stream
directly. The helper below wires up a fresh `BacktestEngine` for one run.

In [8]:
def _build_1min_bars(df: pd.DataFrame, instrument, price_type: str = PRICE_TYPE) -> tuple[BarType, list[Bar]]:
    bt_1min = BarType.from_str(f"{instrument.id}-1-MINUTE-{price_type}-EXTERNAL")
    bars = BarDataWrangler(bt_1min, instrument).process(df)
    return bt_1min, bars


def run_strategy(strategy_factory) -> None:
    """Spin up a BacktestEngine, load 1-min bars, attach a strategy via the factory.

    `strategy_factory(instrument, bar_type_1min)` must return a `Strategy` instance.
    """
    instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)
    bt_1min, bars = _build_1min_bars(df_1min, instrument)

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("AGG-DEMO-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    engine.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    engine.add_instrument(instrument)
    engine.add_data(bars)

    strategy = strategy_factory(instrument, bt_1min)
    engine.add_strategy(strategy)
    engine.run()
    engine.dispose()

### Run Strategy 1 — Nautilus internal aggregation

In [9]:
def make_internal_strategy(instrument, bt_1min: BarType) -> NautilusInternalStrategy:
    bt_5min_internal = BarType.from_str(
        f"{instrument.id}-5-MINUTE-{PRICE_TYPE}-INTERNAL@1-MINUTE-EXTERNAL"
    )
    cfg = NautilusInternalConfig(
        instrument_id=instrument.id,
        bar_type=bt_5min_internal,
        output_csv=str(OUTPUT_NAUTILUS),
    )
    return NautilusInternalStrategy(cfg)


run_strategy(make_internal_strategy)
df_5min_nautilus = pd.read_csv(OUTPUT_NAUTILUS)
print(f"nautilus internal: {len(df_5min_nautilus)} rows → {OUTPUT_NAUTILUS}")
df_5min_nautilus.head()

nautilus internal: 246 rows → d:\mcube_test_version_1_updated\m-cube_version1\eurusd_5min_nautilus.csv


,timestamp,open,high,low,close,volume
0,2015-01-01T22:00:00+00:00,1.21066,1.21067,1.21059,1.21059,5.0
1,2015-01-01T22:05:00+00:00,1.21060,1.21071,1.21045,1.21046,99.0
2,2015-01-01T22:10:00+00:00,1.21045,1.21047,1.21033,1.21038,37.0
3,2015-01-01T22:15:00+00:00,1.21037,1.21082,1.21029,1.21050,150.0
4,2015-01-01T22:20:00+00:00,1.21050,1.21057,1.21008,1.21011,137.0


### Run Strategy 2 — Custom aggregator + derived features

In [10]:
def make_custom_strategy(instrument, bt_1min: BarType) -> CustomAggregationStrategy:
    cfg = CustomAggregationConfig(
        instrument_id=instrument.id,
        bar_type=bt_1min,
        output_csv=str(OUTPUT_CUSTOM),
        target_minutes=5,
    )
    return CustomAggregationStrategy(cfg)


run_strategy(make_custom_strategy)
df_5min_custom = pd.read_csv(OUTPUT_CUSTOM)
print(f"custom aggregator: {len(df_5min_custom)} rows → {OUTPUT_CUSTOM}")
df_5min_custom.head()

custom aggregator: 247 rows → d:\mcube_test_version_1_updated\m-cube_version1\eurusd_5min_custom_features.csv


,timestamp,open,high,low,close,volume,n_ticks,range,body,upper_wick,lower_wick,log_return,pct_return,typical_price,vwap,tr,atr_14,ema_9,ema_21,rsi_14
0,2015-01-01T22:00:00+00:00,1.21066,1.21067,1.21059,1.21059,5.0,1,0.00008,0.00007,0.00001,0.00000,NaN,NaN,1.210617,1.210617,0.00008,0.000080,1.210590,1.210590,NaN
1,2015-01-01T22:05:00+00:00,1.21060,1.21071,1.21045,1.21046,99.0,5,0.00026,0.00014,0.00011,0.00001,-0.000107,-0.000107,1.210540,1.210544,0.00026,0.000093,1.210564,1.210578,0.000000
2,2015-01-01T22:10:00+00:00,1.21045,1.21047,1.21033,1.21038,37.0,5,0.00014,0.00007,0.00002,0.00005,-0.000066,-0.000066,1.210393,1.210504,0.00014,0.000096,1.210527,1.210560,0.000000
3,2015-01-01T22:15:00+00:00,1.21037,1.21082,1.21029,1.21050,150.0,5,0.00053,0.00013,0.00032,0.00008,0.000099,0.000099,1.210537,1.210521,0.00053,0.000127,1.210522,1.210555,39.167361
4,2015-01-01T22:20:00+00:00,1.21050,1.21057,1.21008,1.21011,137.0,5,0.00049,0.00039,0.00007,0.00003,-0.000322,-0.000322,1.210253,1.210435,0.00049,0.000153,1.210439,1.210514,16.520334


### Sanity check

On overlapping timestamps the OHLCV columns from the two paths should be
bit-identical — the only material difference is that the custom strategy adds
the trailing partial window (flushed in `on_stop`), whereas the Nautilus engine
stops emitting once the source data is exhausted.

In [11]:
a = df_5min_nautilus.set_index("timestamp")[["open", "high", "low", "close", "volume"]]
b = df_5min_custom.set_index("timestamp")[["open", "high", "low", "close", "volume"]]
common = a.index.intersection(b.index)
print(f"nautilus rows: {len(a)}   custom rows: {len(b)}   overlap: {len(common)}")
if len(common):
    print("max abs diff on overlap:")
    print((a.loc[common] - b.loc[common]).abs().max())

nautilus rows: 246   custom rows: 247   overlap: 246
max abs diff on overlap:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64
